# HRP Database Setup

This notebook does the following:
1. Loading health monitoring data from Excel files
2. Setting up a local SQLite database for the data storage
3. Creating a normalized schema out of measurements, seniors, and other tables.

**Summary of Results:**
- **Seniors**: 13,317 unique seniors
- **Measurements**: 72,654,799 measurements for different types
- **Medical Info**: 8,220 seniors with disease & medication data
- **Diseases**: 161 unique diseases
- **Medications**: 1,713 unique medications
- **SOS Alerts**: 5,420 alert records
- **Storage**: Local SQLite database

In [1]:
import sys
import os
import time
import sqlite3
from pathlib import Path
import warnings

import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.utils.load_data import load_all_data

warnings.filterwarnings('ignore')

## Section 1: Load and Inspect Raw Excel Data

Load the new collection: three measurement files, medical/diseases, seniors demographics (gender, birthdate, age), and SOS alerts. Inspect structure, types, and basic quality.

In [2]:
# Define paths
raw_data_dir = Path("../data/raw/HRP_new")
measurement_files = [
    raw_data_dir / "data_202512221122-01-09.xlsx",
    raw_data_dir / "data_202512221231-16-23.xlsx",
    raw_data_dir / "data_202512221344-24-30.xlsx",
]
med_file = raw_data_dir / "Med&Diseases_202512221410.xlsx"
demo_file = raw_data_dir / "SeniorGenderAge_202512221409.xlsx"
sos_file = raw_data_dir / "SOS_202512221411.xlsx"

# Ensure all files exist
for p in measurement_files + [med_file, demo_file, sos_file]:
    assert p.exists(), f"Missing file: {p}"

In [3]:
# Load Medications & Disease Data
df_medical = pd.read_excel(med_file, engine="openpyxl")
df_medical.shape

(8224, 3)

In [4]:
df_medical.columns

Index(['seniorID', 'diseaseNames', 'medicineNames'], dtype='object')

In [5]:
df_medical.dtypes

seniorID          int64
diseaseNames     object
medicineNames    object
dtype: object

In [6]:
df_medical.head()

,seniorID,diseaseNames,medicineNames
0,2875,"Osteoporoza,Nadciśnienie tętnicze,Arytmia serc...","Acard,Emanera,Agen,Concor,Valzek"
1,3755,"Miażdzyca,Osteoporoza","Gensulin,Beto,Furosemidum,Amlopin,Zahron,Berod..."
2,3762,"Cukrzyca,Niedoczynnośc tarczycy,Niedoczynnośc ...","Letrox,Diosminex,Valsacor,Metformax,Bibloc,Pol..."
3,3805,"Stomia,Niedosłuch,Skolioza","Pregabalin,Staveran,Neurovit"
4,4367,"Miażdżyca kończyn dolnych,Niewydolnośc układu ...","Allupol,Cipropol,Eliquis,Ezehron,Areplex"


In [7]:
df_medical.isnull().sum()

seniorID         0
diseaseNames     0
medicineNames    0
dtype: int64

In [8]:
# Load Demographics Data
df_demo_raw = pd.read_excel(demo_file, engine="openpyxl")
df_demo_raw.shape

(13317, 4)

In [9]:
df_demo_raw.columns

Index(['seniorID', 'gender', 'birthDate', 'age'], dtype='object')

In [10]:
df_demo_raw.dtypes

seniorID       int64
gender        object
birthDate     object
age          float64
dtype: object

In [11]:
df_demo_raw.isnull().sum()

seniorID       0
gender         0
birthDate    249
age          249
dtype: int64

In [12]:
# Load SOS Alerts
df_sos = pd.read_excel(sos_file, engine="openpyxl")
df_sos.shape

(5420, 3)

In [13]:
df_sos.columns

Index(['seniorID', 'alertDate', 'sosNote'], dtype='object')

In [14]:
df_sos.dtypes

seniorID              int64
alertDate    datetime64[ns]
sosNote              object
dtype: object

In [15]:
df_sos.head()

,seniorID,alertDate,sosNote
0,3205,2025-11-30 17:07:51,Alarm przypadkowy
1,3221,2025-11-25 19:38:36,Alarm przypadkowy
2,3275,2025-11-14 15:01:34,Alarm przypadkowy
3,3279,2025-11-09 11:58:31,Alarm przypadkowy
4,3283,2025-11-17 18:44:32,Alarm przypadkowy


In [16]:
df_sos.isnull().sum()

seniorID      0
alertDate     0
sosNote      85
dtype: int64

In [17]:
# Measurement data sheets (inspect first file)
xls = pd.ExcelFile(measurement_files[0])
sheet_names = xls.sheet_names
print(f"Total sheets in first file: {len(sheet_names)}")
print(f"Sheet names: {sheet_names}\n")

Total sheets in first file: 26
Sheet names: ['expdata', 'expdata#1', 'expdata#2', 'expdata#3', 'expdata#4', 'expdata#5', 'expdata#6', 'expdata#7', 'expdata#8', 'expdata#9', 'expdata#10', 'expdata#11', 'expdata#12', 'expdata#13', 'expdata#14', 'expdata#15', 'expdata#16', 'expdata#17', 'expdata#18', 'expdata#19', 'expdata#20', 'expdata#21', 'expdata#22', 'expdata#23', 'expdata#24', 'expdata#25']



In [18]:
for i, sheet_name in enumerate(sheet_names[:2]):
    df = pd.read_excel(measurement_files[0], sheet_name=sheet_name, nrows=5, engine="openpyxl")
    print(f"\nSheet '{sheet_name}':")
    print(f"    Col 0 (seniorID): {df.iloc[:, 0].values[:2]}")
    print(f"    Col 1 (value): {df.iloc[:, 1].values[:2]}")
    print(f"    Col 2 (sbp): {df.iloc[:, 2].values[:2]}")
    print(f"    Col 3 (dbp): {df.iloc[:, 3].values[:2]}")
    print(f"    Col 4 (date): {df.iloc[:, 4].values[:2]}")
    print(f"    Col 5 (type): {df.iloc[:, 5].values[:2]}")


Sheet 'expdata':
    Col 0 (seniorID): [48129 48427]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-03T23:16:12.000000000' '2025-11-03T23:16:12.000000000']
    Col 5 (type): ['Temperature' 'Temperature']

Sheet 'expdata#1':
    Col 0 (seniorID): [48313 42183]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-04T07:59:41.000000000' '2025-11-04T07:59:41.000000000']
    Col 5 (type): ['Temperature' 'Temperature']


## Section 2: Store Data in SQLite Database

Use the pipeline from `src.utils.load_data` to load demographics, measurements, medical info, and alerts into SQLite.

In [ ]:
# Run end-to-end load using the following pipeline
# Use streaming to avoid large memory usage and commit in batches
# Set fresh_start=True to rebuild the DB and process all sheets from scratch
load_all_data(data_dir=raw_data_dir, fresh_start=False, streaming=True, batch_rows=100_000, resume=True)

INFO:src.utils.database:Database initialized at c:\Users\eldar\Projects\AI-CVD\db\hrp_data.db
INFO:src.utils.load_data:Loading seniors demographics from ..\data\raw\HRP_new\SeniorGenderAge_202512221409.xlsx


INFO:src.utils.load_data:Loaded 13317 senior demographic rows
INFO:src.utils.load_data:Upserted 13317 seniors with demographics
INFO:src.utils.load_data:Streaming measurements from ..\data\raw\HRP_new\data_202512221122-01-09.xlsx
INFO:src.utils.load_data:  expdata: +100,000 (total 100,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 200,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 300,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 400,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 500,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 600,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 700,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 800,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 900,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 1,000,000)
INFO:src.utils.load_data:  expdata: +48,575 (total 1,048,575)
INFO:src.utils.load_data:✓ Finished sheet 'expdata'
INFO:src.utils.load_data:  expdata#1: +100,0


DATABASE SUMMARY
seniors.......................          13,317
measurements..................      72,654,799
alerts........................           5,420
medical_info (raw)............           8,220
diseases......................             161
medicines.....................           1,713
senior_diseases...............          49,444
senior_medicines..............          48,304


In [20]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

In [21]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [22]:
print(f"Database initialized at {db_path.as_posix()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

Database initialized at ../db/hrp_data.db
Database size: 14325904.0 KB


In [23]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables created: {[t[0] for t in tables]}")


Tables created: ['seniors', 'measurements', 'sqlite_sequence', 'medical_info', 'diseases', 'medicines', 'senior_diseases', 'senior_medicines', 'alerts', 'ingestion_state']


# Section 3: Verify/Inspect Data Integrity

In [24]:
# Pull measurements 
df_measurements = pd.read_sql("SELECT * FROM measurements LIMIT 100000", conn)
print(df_measurements.shape)
df_measurements.head()

(100000, 7)


,id,senior_id,value,sbp,dbp,date,type
0,1,48129,36.3,None,None,2025-11-03 23:16:12,Temperature
1,2,48427,36.6,None,None,2025-11-03 23:16:12,Temperature
2,3,45310,36.3,None,None,2025-11-03 23:16:12,Temperature
3,4,29421,36.6,None,None,2025-11-03 23:16:12,Temperature
4,5,45316,36.9,None,None,2025-11-03 23:16:12,Temperature


In [25]:
# Medical information counts
counts_med = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM medical_info", conn)
counts_med

,n_rows,n_seniors
0,8220,8220


In [26]:
# Alerts counts
counts_alerts = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM alerts", conn)
counts_alerts

,n_rows,n_seniors
0,5420,3062


In [27]:
# Preview alerts
pd.read_sql("SELECT * FROM alerts LIMIT 5", conn)

,alert_id,senior_id,alert_date,sos_note
0,1,3205,2025-11-30 17:07:51,Alarm przypadkowy
1,2,3221,2025-11-25 19:38:36,Alarm przypadkowy
2,3,3275,2025-11-14 15:01:34,Alarm przypadkowy
3,4,3279,2025-11-09 11:58:31,Alarm przypadkowy
4,5,3283,2025-11-17 18:44:32,Alarm przypadkowy


In [ ]:
# Measurement type distribution
measure_type_counts = pd.read_sql(
    "SELECT type, COUNT(*) AS cnt FROM measurements GROUP BY type ORDER BY cnt DESC",
    conn,
)
measure_type_counts

,type,cnt
0,Heartrate,16429481
1,BloodPressure,16429476
2,Temperature,16359953
3,Saturation,13121312
4,Steps,10314577


## Section 4: Query and Validate Stored Data

Execute SQL queries to retrieve data and perform basic analysis to confirm database functionality.

### Example 1: Get all measurements for a specific type

In [29]:
query1 = """
    SELECT senior_id, value, date, type
    FROM measurements
    WHERE type = 'Heartrate'
    ORDER BY date
    LIMIT 10
"""

In [30]:
df_example1 = pd.read_sql(query1, conn)
df_example1

,senior_id,value,date,type
0,20307,88.0,2025-11-01 00:00:10,Heartrate
1,8721,62.0,2025-11-01 00:00:10,Heartrate
2,41788,104.0,2025-11-01 00:00:10,Heartrate
3,44626,85.0,2025-11-01 00:00:10,Heartrate
4,44759,65.0,2025-11-01 00:00:10,Heartrate
5,44228,50.0,2025-11-01 00:00:10,Heartrate
6,43633,75.0,2025-11-01 00:00:10,Heartrate
7,30312,73.0,2025-11-01 00:00:10,Heartrate
8,28661,67.0,2025-11-01 00:00:10,Heartrate
9,32587,54.0,2025-11-01 00:00:11,Heartrate


### Example 2: Aggregate statistics by measurement type

In [31]:
query2 = """
    SELECT 
        type,
        COUNT(*) as measurement_count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        AVG(value) as avg_value,
        MIN(value) as min_value,
        MAX(value) as max_value,
        ROUND(AVG(value), 2) as mean
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY measurement_count DESC
"""

In [ ]:
df_example2 = pd.read_sql(query2, conn)
df_example2

### Example 3: Get measurements for a specific senior

In [ ]:
sample_senior_id = int(df_measurements.iloc[0]["senior_id"])
query3 = """
    SELECT senior_id, value, sbp, dbp, date, type
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date DESC
    LIMIT 10
"""

In [ ]:
df_example3 = pd.read_sql(query3, conn, params=[sample_senior_id])
df_example3

,senior_id,value,sbp,dbp,date,type
0,36280,99.0,NaN,NaN,2025-11-15 23:56:11,Saturation
1,36280,NaN,138.0,89.0,2025-11-15 23:56:11,BloodPressure
2,36280,64.0,NaN,NaN,2025-11-15 23:56:11,Heartrate
3,36280,36.6,NaN,NaN,2025-11-15 23:56:11,Temperature
4,36280,99.0,NaN,NaN,2025-11-15 23:46:11,Saturation
5,36280,NaN,134.0,89.0,2025-11-15 23:46:11,BloodPressure
6,36280,64.0,NaN,NaN,2025-11-15 23:46:11,Heartrate
7,36280,36.8,NaN,NaN,2025-11-15 23:46:11,Temperature
8,36280,99.0,NaN,NaN,2025-11-15 23:36:11,Saturation
9,36280,NaN,142.0,92.0,2025-11-15 23:36:11,BloodPressure


### Example 4: Query performance test

In [ ]:
start = time.time()
query4 = "SELECT * FROM measurements WHERE type = 'Heartrate' LIMIT 1000"
df_example4 = pd.read_sql(query4, conn)
elapsed = time.time() - start

In [ ]:
print(f"  Retrieved {len(df_example4)} rows in {elapsed:.4f} seconds")

  Retrieved 1000 rows in 0.0254 seconds


### Example 4: Blood Pressure Analysis

In [ ]:
query5 = """
    SELECT senior_id, sbp, dbp, date, type
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
    ORDER BY date DESC
    LIMIT 10
"""

In [ ]:
df_example5 = pd.read_sql(query5, conn)
df_example5

,senior_id,sbp,dbp,date,type
0,9263,128.0,55.0,2025-11-15 23:59:53,BloodPressure
1,50494,112.0,89.0,2025-11-15 23:59:38,BloodPressure
2,26794,120.0,78.0,2025-11-15 23:59:33,BloodPressure
3,4043,151.0,57.0,2025-11-15 23:59:32,BloodPressure
4,48782,133.0,83.0,2025-11-15 23:59:32,BloodPressure
5,30983,126.0,72.0,2025-11-15 23:59:32,BloodPressure
6,49493,124.0,84.0,2025-11-15 23:59:31,BloodPressure
7,47346,119.0,75.0,2025-11-15 23:59:31,BloodPressure
8,21960,112.0,70.0,2025-11-15 23:59:31,BloodPressure
9,40467,132.0,75.0,2025-11-15 23:59:31,BloodPressure


### Example 5: Get Blood Pressure Statistics

In [ ]:
query5_stats = """
    SELECT 
        COUNT(*) as bp_measurements,
        COUNT(DISTINCT senior_id) as seniors_with_bp,
        AVG(sbp) as avg_systolic,
        AVG(dbp) as avg_diastolic,
        MIN(sbp) as min_systolic,
        MAX(sbp) as max_systolic,
        MIN(dbp) as min_diastolic,
        MAX(dbp) as max_diastolic
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
"""

In [ ]:
df_bp_stats = pd.read_sql(query5_stats, conn)
df_bp_stats

,bp_measurements,seniors_with_bp,avg_systolic,avg_diastolic,min_systolic,max_systolic,min_diastolic,max_diastolic
0,4074731,11762,129.392577,78.623184,68.0,212.0,23.0,156.0
